# Feature engineering on landsat
This notebooks take the bands returned by landsat and calculates more features

## Environment preparation

In [ ]:
!pip install uv
!uv pip install  -r ../requirements.txt 

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Data manipulation and analysis
import numpy as np
import pandas as pd

# Retry
import time
from pystac_client.exceptions import APIError

# Importing function
import sys
import os
sys.path.append(os.path.abspath('..'))
from utils import save_df, landsat_bands_of_interest

In [ ]:
# Processing
eps = 1e-10

# Path
save_path = 'snow://workspace/USER$.PUBLIC."waterscript-ey-data-science-challenge"/versions/live/data/processed/'

In [ ]:
#Note

# The Landsat data extraction process for all 9,319 locations typically requires more than 7 hours when
# executed in a single run. During long executions, you may occasionally encounter API limits, timeout errors,
# or request failures. To avoid these interruptions, we recommend running the extraction in smaller batches.

# We have already executed the full extraction for all 9,319 locations and saved the output
# to **landsat_features_training.csv**

def process_df(destination_df, original_df):
    '''
    Calculates spectral indices and merges metadata from the original dataset.
    Indices documentation: https://docs.google.com/document/d/1VOdy3sTt4UJxjBM_zrndnOT1DKKQ_yHdgqNfwF0iKn4/edit?usp=sharing

    Calculates indices:
    - NDMI (Normalized Difference Moisture Index): Measures vegetation water content. Formula: (NIR - SWIR16) / (NIR + SWIR16).
    - MNDWI (Modified Normalized Difference Water Index): Highlights open water features. Formula: (Green - SWIR16) / (Green + SWIR16).
    - NDTI (Normalized Difference Turbidity Index): Estimates water turbidity/cloudiness. Formula: (Red - Green) / (Red + Green).
    - NDSSI (Normalized Difference Suspended Sediment Index): Helps to detect thin sediments. Formula: (Blue - NIR) / (Blue + NIR).
    - Chlorophyll_Proxy: Estimates the F quantity. Formula: NIR / Green.
    - Red_Blue_Ratio: Helps to infer alcalinity. Formula: Red / Blue.
    - SI (Salinity Index): Measures the dissolved salts. Formula: np.sqrt(Blue * Red)
    - NDVI (Normalized Difference Vegetation Index): Detects near vegetation. Formula (NIR - Red) / (NIR + Red).
    '''
    # Corrects the bands as the getting_started notebook shows (USGS)
    for band in landsat_bands_of_interest:
        if destination_df[band].mean() > 1:
            print(f"Corrigindo escala da banda {band}...")
            destination_df[band] = (destination_df[band] * 0.0000275) - 0.2
            
    # NDMI
    destination_df['NDMI'] = (
        (destination_df['nir08'] - destination_df['swir16'])
        / (destination_df['nir08'] + destination_df['swir16'] + eps)
    )
    
    # MNDWI
    destination_df['MNDWI'] = (
        (destination_df['green'] - destination_df['swir16'])
        / (destination_df['green'] + destination_df['swir16'] + eps)
    )
    
    # NDTI
    destination_df['NDTI'] = (
        (destination_df['red'] - destination_df['green'])
        / (destination_df['red'] + destination_df['green'] + eps)
    )
    
    # NDSSI
    destination_df['NDSSI'] = (
        (destination_df['blue'] - destination_df['nir08'])
        / (destination_df['blue'] + destination_df['nir08'] + eps)
    )

    # Chlorophyll_Proxy
    destination_df['Chlorophyll_Proxy'] = (
        destination_df['nir08'] / (destination_df['green'] + eps)
    )

    # Red_Blue_Ratio
    destination_df['Red_Blue_Ratio'] = (
        destination_df['red'] / (destination_df['blue'] + eps)
    )

    # Salinity Index
    destination_df['SI'] = np.sqrt(
        destination_df['blue'] * destination_df['red']
    )

    # NDVI
    destination_df['NDVI'] = (
        (destination_df['nir08'] - destination_df['red'])
        / (destination_df['nir08'] + destination_df['red'] + eps)
    )

    # Finalizing df
    destination_df['Latitude'] = original_df['Latitude']
    destination_df['Longitude'] = original_df['Longitude']
    destination_df['Sample Date'] = original_df['Sample Date']
    cols_to_keep = [
        'Latitude', 'Longitude', 'Sample Date', 
        'NDMI', 'MNDWI', 'NDTI', 'NDSSI', 'Chlorophyll_Proxy', 'Red_Blue_Ratio', 'SI', 'NDVI'
    ] + landsat_bands_of_interest
    
    return destination_df[cols_to_keep]

In [ ]:
# Df for training
water_quality_df = (
    pd.read_csv('../data/raw/water_quality_training_dataset.csv')
)

landsat_train_features_base = (
    pd.read_csv('../data/intermediate/landsat_train_features_base.csv')
)

# Df for testing
validation_df = (
    pd.read_csv('../data/raw/submission_template.csv')
)

landsat_val_features_base = (
    pd.read_csv('../data/intermediate/landsat_val_features_base.csv')
)

## Data transformation

In [ ]:
landsat_train_features_base_processed = process_df(landsat_train_features_base, water_quality_df)
landsat_val_features_base_processed = process_df(landsat_val_features_base, validation_df)

## Data saving

In [ ]:
save_df(landsat_train_features_base_processed, 'landsat_train_features', save_path)
save_df(landsat_val_features_base_processed, 'landsat_val_features', save_path)